# Análisis interno de ESM-2: Proyecto Completo

Trabajo de Segundo Corte - Ciencia de Datos  
Universidad de Pamplona - Ingeniería de Sistemas

**Autor:** Anderson González (@Albonire)  
**Modelo:** facebook/esm2_t6_8M_UR50D

Este trabajo analiza por dentro el modelo ESM-2, un transformer científico para proteínas.
No se entrena el modelo, no se hace fine-tuning ni se construye un chatbot. El objetivo es

ejecutar el modelo preentrenado y observar qué ocurre internamente cuando recibe una
secuencia de proteína.

El notebook integra las 7 actividades:
1. Marco teórico esencial.
2. Carga y configuración del modelo.
3. Ejecución y extracción de formas internas.
4. Inspección del código fuente real.
5. Embeddings y similitud entre secuencias.
6. PCA y matrices de atención.
7. Masked Language Modeling.

Al ejecutarse, el notebook genera automáticamente los assets en la carpeta `outputs/`.

## 1. Teoría esencial (lo mínimo)

**ESM-2 es encoder-only.** Lee toda la secuencia a la vez, mirando ambos lados de cada
posición. Es distinto a GPT, que es decoder-only y solo mira hacia atrás para generar texto.
ESM-2 no genera nada: solo entiende y produce representaciones.

**Concepto clave — embedding inicial vs hidden state:**
- **Embedding inicial:** el token solo, sin contexto. Su primer vector.
- **Hidden state:** el token ya procesado, con información del resto de la secuencia.

Esa diferencia se va a ver reflejada en el código y en los resultados.

**Self-attention (Q, K, V):**
- Q (Query): qué busca cada token.
- K (Key): cómo se presenta cada token.
- V (Value): qué información aporta cada token.

Fórmula: softmax(Q·Kᵀ / √dk)·V. En multi-head attention se ejecutan varias atenciones en paralelo.

## 2. Instalación del entorno

In [ ]:
%pip install -q torch transformers matplotlib scikit-learn pandas tabulate

## 3. Importación de librerías y configuración

In [ ]:
import os
import inspect
import textwrap
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, EsmModel, EsmForMaskedLM
from transformers.models.esm import modeling_esm

MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
SEQUENCES = {
    "Original": "MKTAYIAKQRQISFVKSHFSRQDILD",
    "Mutada":   "MKTAFIAKQRQISFVKSHFSRQDILD",
    "Alterada": "DLIDQRSFHSSKVFSIQRQKAIYATKM",
}

# Carpeta donde se guardan los assets (imagenes y csv) para el informe
os.makedirs("outputs", exist_ok=True)
print("PyTorch:", torch.__version__)
print("Secuencias a analizar:", list(SEQUENCES.keys()))

## 4. Carga y configuración del modelo

Se cargan dos versiones: `EsmModel` para hidden states y atenciones, y `EsmForMaskedLM`
para la predicción de tokens. Se usa `attn_implementation="eager"` dentro de un try-except
porque algunas versiones optimizadas no devuelven las matrices de atención.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

try:
    model = EsmModel.from_pretrained(MODEL_NAME, attn_implementation="eager")
    print("EsmModel cargado con attn_implementation='eager'.")
except Exception:
    model = EsmModel.from_pretrained(MODEL_NAME)
    print("EsmModel cargado sin attn_implementation.")
model.eval()

mlm_model = EsmForMaskedLM.from_pretrained(MODEL_NAME)
mlm_model.eval()
print("Modelos cargados correctamente.")

### 4.1 Configuración del modelo

In [ ]:
cfg = model.config
print("Modelo:", MODEL_NAME)
print("hidden_size:", cfg.hidden_size, " -> cada token se representa con", cfg.hidden_size, "valores")
print("num_hidden_layers:", cfg.num_hidden_layers, "capas")
print("num_attention_heads:", cfg.num_attention_heads, "cabezas de atención")
print("vocab_size:", cfg.vocab_size, "tokens posibles")

## 5. Ejecución y formas internas

La función `analyze` recibe una secuencia, la tokeniza y corre el modelo con
`output_hidden_states=True` y `output_attentions=True`. Estos parámetros le indican al
modelo que devuelva los estados internos y las atenciones, no solo el resultado final.

Para cada secuencia se extraen: tokens, IDs, forma del last_hidden_state, número de hidden
states y forma de las atenciones.

In [ ]:
def analyze(name, seq):
    inputs = tokenizer(seq, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, output_attentions=True)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    ids = inputs["input_ids"][0].tolist()
    att_ok = out.attentions is not None

    print("="*70)
    print(f"Secuencia: {name}  ->  {seq}")
    print(f"Longitud (sin tokens especiales): {len(seq)}")
    print(f"Tokens: {tokens}")
    print(f"IDs: {ids}")
    print(f"input_ids shape: {tuple(inputs['input_ids'].shape)}")
    print(f"last_hidden_state shape: {tuple(out.last_hidden_state.shape)}")
    print(f"Número de hidden_states: {len(out.hidden_states)}")
    print(f"¿Atenciones disponibles?: {att_ok}")
    if att_ok:
        print(f"Número de tensores de atención: {len(out.attentions)}")
        print(f"Forma atención capa 0: {tuple(out.attentions[0].shape)}")
    return inputs, out

results = {n: analyze(n, s) for n, s in SEQUENCES.items()}

### 5.1 Tabla resumen de las tres secuencias

Una tabla con pandas que junta los datos de las 3 secuencias para compararlos de un vistazo.
La forma del last_hidden_state es (1, 28, 320): 1 secuencia, 28 tokens (incluyendo especiales),
320 valores por token.

In [ ]:
rows = []
for n, (inp, out) in results.items():
    rows.append({
        "tipo": n,
        "longitud": len(SEQUENCES[n]),
        "input_ids_shape": tuple(inp["input_ids"].shape),
        "last_hidden_state_shape": tuple(out.last_hidden_state.shape),
        "num_hidden_states": len(out.hidden_states),
        "attentions": out.attentions is not None,
        "att_shape": tuple(out.attentions[0].shape) if out.attentions is not None else None,
    })
pd.DataFrame(rows)

## 6. Inspección del código fuente

La guía pide no solo nombrar los componentes, sino mostrarlos en el código real.

### 6.1 Arquitectura general con print(model)

In [ ]:
print(model)

### 6.2 Tabla de componentes

| Componente | Clase | Qué hace |
|---|---|---|
| Tokenizer | EsmTokenizer | Convierte aminoácidos y tokens especiales en IDs |
| Embeddings | EsmEmbeddings | Convierte IDs en vectores iniciales |
| Self-attention | EsmSelfAttention | Calcula Q, K, V y pesos de atención |
| Multi-head | dentro de EsmSelfAttention | Varias cabezas en paralelo |
| LayerNorm/residual | EsmLayer, EsmSelfOutput | Estabiliza y preserva información |
| Feed-forward | EsmIntermediate, EsmOutput | MLP por posición |
| MLM head | EsmLMHead | Logits para predecir tokens ocultos |

### 6.3 Fragmentos del código fuente real

La función `inspect_component` usa la librería `inspect` de Python para sacar el código
fuente real de cada clase. En `EsmSelfAttention` se ven literalmente las líneas que crean
query, key y value.

In [ ]:
def inspect_component(title, obj, max_lines=40):
    print("="*70)
    print(title)
    print("="*70)
    try:
        print("Archivo:", inspect.getfile(obj))
    except Exception as e:
        print("No se pudo obtener archivo:", e)
    try:
        lines = inspect.getsource(obj).splitlines()
        print(textwrap.indent("\n".join(lines[:max_lines]), "    "))
        if len(lines) > max_lines:
            print(f"    ... (recortado, total {len(lines)} líneas)")
    except Exception as e:
        print("No se pudo obtener código:", e)
    print()

inspect_component("EsmEmbeddings (IDs -> vectores)", modeling_esm.EsmEmbeddings, 35)
inspect_component("EsmSelfAttention (aquí están query, key, value)", modeling_esm.EsmSelfAttention, 55)
inspect_component("EsmLayer (capa completa)", modeling_esm.EsmLayer, 40)
inspect_component("EsmLMHead (predice tokens)", modeling_esm.EsmLMHead, 35)

## 7. Embeddings y similitud

Dos funciones:
- `get_residue_emb` saca el embedding de cada aminoácido. Usa `[0, 1:-1, :]` para excluir
  los tokens especiales de inicio y fin.
- `get_global_emb` promedia esos vectores en uno solo que representa toda la proteína.

In [ ]:
def get_residue_emb(out):
    # Excluye token inicial (<cls>) y final (<eos>)
    return out.last_hidden_state[0, 1:-1, :].numpy()

def get_global_emb(out):
    return out.last_hidden_state[0, 1:-1, :].mean(dim=0).numpy()

residue_embs = {n: get_residue_emb(r[1]) for n, r in results.items()}
glob_embs = {n: get_global_emb(r[1]) for n, r in results.items()}

for n in SEQUENCES:
    print(f"{n}: embedding por residuo {residue_embs[n].shape} | global {glob_embs[n].shape}")

### 7.1 Similitud coseno entre embeddings globales

Se compara las 3 secuencias con `cosine_similarity` de scikit-learn. La tabla se guarda en
`outputs/similitud_coseno.csv`. Se espera que Original vs Mutada sea alta (~0.99, solo cambia
una letra) y Original vs Alterada sea más baja (cambia el orden completo). Esto demuestra que
el modelo entiende el orden de la secuencia, no solo las letras sueltas.

In [ ]:
names = list(glob_embs.keys())
matrix = np.array([glob_embs[n] for n in names])
sim = cosine_similarity(matrix)
df_sim = pd.DataFrame(sim, index=names, columns=names)
df_sim.to_csv("outputs/similitud_coseno.csv")
print("Tabla guardada en outputs/similitud_coseno.csv")
df_sim

## 8. PCA: visualización 2D

El PCA reduce los 320 valores de cada aminoácido a 2 dimensiones para poder graficarlos.
Cada punto es un aminoácido, coloreado según a qué secuencia pertenece. Es una proyección
matemática, no una representación biológica real. La imagen se guarda en
`outputs/pca_embeddings.png`.

In [ ]:
all_vecs, all_labels = [], []
for n in names:
    v = residue_embs[n]
    all_vecs.append(v)
    all_labels += [n]*v.shape[0]
all_vecs = np.vstack(all_vecs)

coords = PCA(n_components=2).fit_transform(all_vecs)

plt.figure(figsize=(8,6))
for n in names:
    idx = [i for i,l in enumerate(all_labels) if l==n]
    plt.scatter(coords[idx,0], coords[idx,1], label=n, alpha=0.7)
plt.title("PCA de embeddings por residuo")
plt.xlabel("Componente 1"); plt.ylabel("Componente 2")
plt.legend(); plt.tight_layout()
plt.savefig("outputs/pca_embeddings.png", dpi=150)
print("Imagen guardada en outputs/pca_embeddings.png")
plt.show()

## 9. Matrices de atención

La función `plot_att` dibuja el mapa de calor. Cada cuadro [i,j] muestra cuánto atiende el
token i al token j. Se muestran varias: capa temprana, capa profunda, dos cabezas distintas
de la misma capa, y comparación original vs mutada.

Las gráficas se guardan automáticamente en `outputs/` con `plt.savefig`.

**Importante:** estos patrones NO prueban nada biológico. Son correlaciones estadísticas,
no causalidad.

In [ ]:
def plot_att(att, tokens, title, fname):
    plt.figure(figsize=(8,7))
    plt.imshow(att, aspect="auto")
    plt.title(title)
    plt.xticks(range(len(tokens)), tokens, rotation=90)
    plt.yticks(range(len(tokens)), tokens)
    plt.colorbar(label="peso de atención")
    plt.tight_layout()
    plt.savefig(f"outputs/{fname}", dpi=150)
    plt.show()

inp_o, out_o = results["Original"]
tokens_o = tokenizer.convert_ids_to_tokens(inp_o["input_ids"][0])

if out_o.attentions is not None:
    n_layers = len(out_o.attentions)
    n_heads = out_o.attentions[0].shape[1]
    print(f"Capas: {n_layers} | Cabezas: {n_heads}")

    # Capa temprana (patrones locales)
    plot_att(out_o.attentions[0][0,0].detach().numpy(), tokens_o,
             "Atención - capa 0, cabeza 0 (temprana)", "atencion_capa_temprana.png")
    # Capa profunda (patrones globales)
    plot_att(out_o.attentions[n_layers-1][0,0].detach().numpy(), tokens_o,
             f"Atención - capa {n_layers-1}, cabeza 0 (profunda)", "atencion_capa_profunda.png")
    # Otra cabeza de la misma capa
    if n_heads >= 2:
        plot_att(out_o.attentions[0][0,1].detach().numpy(), tokens_o,
                 "Atención - capa 0, cabeza 1", "atencion_capa0_cabeza1.png")
else:
    print("Atenciones no disponibles. Documentar el intento y centrar en hidden states.")

### 9.1 Comparación de atención: Original vs Mutada

In [ ]:
inp_m, out_m = results["Mutada"]
tokens_m = tokenizer.convert_ids_to_tokens(inp_m["input_ids"][0])

if out_o.attentions is not None and out_m.attentions is not None:
    plot_att(out_o.attentions[0][0,0].detach().numpy(), tokens_o,
             "Original - capa 0, cabeza 0", "atencion_original_c0h0.png")
    plot_att(out_m.attentions[0][0,0].detach().numpy(), tokens_m,
             "Mutada - capa 0, cabeza 0", "atencion_mutada_capa0_cabeza0.png")
    print("Hay diferencias visibles, confirma que el modelo es sensible al cambio.")

## 10. Masked Language Modeling

Se toma la secuencia original y se reemplaza un aminoácido por el token de máscara con
`tokenizer.mask_token`. Se busca la posición de la máscara, se corre `EsmForMaskedLM` y de
los logits se sacan los 5 tokens más probables con `torch.topk`.

Esto demuestra que ESM-2 aprendió qué aminoácidos suelen ir en ciertos contextos. Como mira
los dos lados del hueco, confirma contexto bidireccional, a diferencia de GPT.

In [ ]:
masked_seq = SEQUENCES["Original"].replace("Y", tokenizer.mask_token, 1)
print("Secuencia original:  ", SEQUENCES["Original"])
print("Secuencia enmascarada:", masked_seq)

inputs_m = tokenizer(masked_seq, return_tensors="pt")
mask_idx = (inputs_m.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

with torch.no_grad():
    logits = mlm_model(**inputs_m).logits

top5 = torch.topk(logits[0, mask_idx, :], 5)
print("\nTop-5 tokens predichos para la posición enmascarada:")
for s, i in zip(top5.values[0], top5.indices[0]):
    tok = tokenizer.convert_ids_to_tokens([i.item()])[0]
    print(f"  {tok}  ->  score: {float(s):.4f}")

print("\nNota: el aminoácido original era 'Y'. Esto NO es generación tipo GPT.")

## 11. Usos reales y limitaciones

### Usos reales
1. Comparación de proteínas mediante embeddings.
2. Análisis de mutaciones.
3. Apoyo a predicción de estructura (ESMFold).
4. Anotación funcional o clasificación posterior.
5. Búsqueda semántica de proteínas en bases vectoriales.
6. Priorización de variantes para ingeniería de proteínas.

### Limitaciones
- La atención no prueba causalidad biológica.
- No reemplaza experimentos de laboratorio.
- No descubre medicamentos automáticamente.
- Refleja sesgos de los datos de entrenamiento.
- Las predicciones requieren validación científica.

## 12. Conclusiones

1. ESM-2 es un transformer encoder-only aplicado a proteínas; no genera, entiende.
2. El last_hidden_state es una representación contextualizada, distinta del embedding inicial.
3. La self-attention se basa en Q, K, V, identificables en el código fuente real.
4. La secuencia mutada conserva alta similitud con la original; la alterada se aleja.
5. El masked language modeling evidencia el carácter bidireccional del modelo.
6. El proyecto es reproducible: genera 5 assets en `outputs/` y se documenta en el informe HTML.

### Assets generados en outputs/
- `similitud_coseno.csv`
- `pca_embeddings.png`
- `atencion_capa_temprana.png`
- `atencion_capa_profunda.png`
- `atencion_capa0_cabeza1.png`
- `atencion_mutada_capa0_cabeza0.png`

## Referencias
- Vaswani et al. (2017). Attention Is All You Need.
- Devlin et al. (2018). BERT.
- Radford et al. (2018). GPT.
- Lin et al. (2023). Evolutionary-scale prediction of atomic-level protein structure.
- Hugging Face Transformers Documentation: ESM.